# Notebook 6: Hybrid Router

Notebook 5 showed each path has its own strength: SQL answers exact questions and vector search handles open ones. This notebook combines them behind a single `ask()`.

The SQL pipeline now lives in `src/text_to_sql.py`, moved unchanged from Notebook 5.

In [ ]:
import json
import re
import sys
from pathlib import Path

import chromadb
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.text_to_sql import (LLM_OPTIONS, MODEL, NO_SQL, NOT_IN_DATA, client, con,
                             numbers_in, run_sql, sql_answer)

# Unload the LLM first: on a machine short of memory the embedding model can't load beside it
client.generate(model=MODEL, keep_alive=0)

CORPUS_PATH = PROJECT_ROOT / "data" / "processed" / "corpus.txt"
with open(CORPUS_PATH, encoding="utf-8") as f:
    sentences = [line.strip() for line in f.readlines()]

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
collection = chromadb.PersistentClient(path=str(PROJECT_ROOT / "chroma_db")).get_collection("f1_corpus")

with open(PROJECT_ROOT / "eval" / "questions.json", encoding="utf-8") as f:
    QUESTIONS = json.load(f)

print(f"Corpus: {len(sentences)} sentences | Chroma: {collection.count()} | Questions: {len(QUESTIONS)}")

## Hybrid retrieval

Dense search matches meaning, so it blurs exact names and years. BM25 matches exact words, which is what this corpus of near-identical sentences needs. Reciprocal rank fusion combines the two using ranks alone, so their different score scales don't matter.

A cross-encoder reranker was left out: it needs a model download this machine has no disk space for.

In [ ]:
def tokenize(text: str) -> list:
    return re.findall(r"[a-z0-9]+", text.lower())


bm25 = BM25Okapi([tokenize(s) for s in sentences])


def dense_search(query: str, k: int) -> list:
    query_vec = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_vec, n_results=k)
    return [int(i) for i in results["ids"][0]]


def bm25_search(query: str, k: int) -> list:
    scores = bm25.get_scores(tokenize(query))
    return sorted(range(len(sentences)), key=lambda i: -scores[i])[:k]


def fused_search(query: str, k: int = 5, pool: int = 50, c: int = 60) -> list:
    """Reciprocal rank fusion: reward sentences ranked highly by either search."""
    fused = {}
    for ranked in (dense_search(query, pool), bm25_search(query, pool)):
        for position, sentence_id in enumerate(ranked):
            fused[sentence_id] = fused.get(sentence_id, 0) + 1 / (c + position + 1)
    return sorted(fused, key=lambda i: -fused[i])[:k]

In [ ]:
# A hit must contain every string. `best` is how many such sentences can exist in the top 5.
RETRIEVAL_CASES = [
    ("How did Max Verstappen perform at Monaco?", ["Max Verstappen", "Monaco Grand Prix"], 5),
    ("Tell me about Lando Norris at the British Grand Prix", ["Lando Norris", "British Grand Prix"], 5),
    ("How did Fernando Alonso do at the Spanish Grand Prix?", ["Fernando Alonso", "Spanish Grand Prix"], 5),
    ("Describe Ferrari in the 2008 constructors championship", ["2008 constructors", "Ferrari"], 1),
    ("What happened to Daniel Ricciardo in the 2018 Monaco Grand Prix?", ["2018 Monaco", "Daniel Ricciardo"], 1),
    ("Tell me about the 2021 Abu Dhabi Grand Prix", ["2021 Abu Dhabi Grand Prix"], 5),
    ("How did Charles Leclerc do at the Italian Grand Prix?", ["Charles Leclerc", "Italian Grand Prix"], 5),
    ("Tell me about Ayrton Senna in the 1988 Japanese Grand Prix", ["1988 Japanese", "Ayrton Senna"], 1),
    ("How did McLaren do in the 1988 constructors championship?", ["1988 constructors", "McLaren"], 1),
    ("Tell me about Lewis Hamilton at Silverstone", ["Lewis Hamilton", "British Grand Prix"], 5),
]

searches = {"dense": dense_search, "bm25": bm25_search, "fused": fused_search}
totals = dict.fromkeys(searches, 0)
for query, needed, best in RETRIEVAL_CASES:
    hits = {name: sum(all(n in sentences[i] for n in needed) for i in fn(query, 5)) for name, fn in searches.items()}
    for name in searches:
        totals[name] += hits[name]
    print(f"{query:<66} {hits}  best {best}")
print(f"\nTOTAL of {sum(c[2] for c in RETRIEVAL_CASES)}: {totals}")

## Keyword router

A router this simple is deliberate: it is fast, free, explainable and testable. An LLM router would add a slow, non-deterministic step in front of every question, using the same small model that already struggles with SQL.

Order matters. Refusal comes first, then open-question openers, because "How did Ferrari do" should not be caught by a fact keyword.

In [ ]:
NARRATIVE = re.compile(r"^\s*(tell me|describe|explain|why|how did|how was|compare|summari[sz]e|what happened)", re.IGNORECASE)
FACT = re.compile(r"\b(how many|who|which|most|average|total|what is the|fastest|oldest|youngest)\b", re.IGNORECASE)


def route(question: str) -> str:
    """refuse: data we don't have. vector: open questions. sql: one exact answer."""
    if NOT_IN_DATA.search(question):
        return "refuse"
    if NARRATIVE.search(question):
        return "vector"
    if FACT.search(question):
        return "sql"
    return "vector"

In [ ]:
labelled = [(q["question"], "sql" if q["truth_sql"] else "refuse") for q in QUESTIONS if q["split"] != "test3"]
labelled += [(case[0], "vector") for case in RETRIEVAL_CASES]
labelled += [
    ("Compare Ferrari and McLaren in 1998", "vector"),
    ("Why did Mercedes dominate in 2014?", "vector"),
]

wrong = [(q, want, route(q)) for q, want in labelled if route(q) != want]
print(f"Routing: {len(labelled) - len(wrong)}/{len(labelled)} correct")
for q, want, got in wrong:
    print(f"  {q} | wanted {want}, got {got}")

## One entry point

If SQL fails, returns nothing or returns a lone zero, `ask()` falls back to retrieval. That zero check targets Notebook 5's worst failure, telling the user Nigel Mansell won no races. The trade-off is that a true answer of zero becomes a refusal, which is the safer mistake.

In [ ]:
def answer_vector(question: str) -> tuple:
    ids = fused_search(question)
    context = "\n".join(f"- {sentences[i]}" for i in ids)
    prompt = f"""Answer the question in one or two sentences using ONLY the context below.
Never guess or infer beyond the context. If the context does not state the answer, reply exactly: Not in the context.

Context:
{context}

Question: {question}
Answer:"""
    response = client.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options=LLM_OPTIONS,
    )
    return response["message"]["content"].strip(), [sentences[i] for i in ids]


def usable(run: dict) -> bool:
    """A failed, empty or zero result is no evidence. A count of 0 usually means a bad filter."""
    if run["sql"] == NO_SQL or run["error"] or run["df"].empty:
        return False
    if run["df"].shape == (1, 1):
        value = run["df"].iloc[0, 0]
        return bool(pd.notna(value)) and value != 0
    return True


def answer_sql_only(question: str) -> str:
    run = run_sql(question)
    return sql_answer(question, run["df"]) if usable(run) else "No usable result from the database."


def ask(question: str) -> dict:
    """The single entry point: choose a path, and fall back to retrieval if SQL gives nothing usable."""
    path = route(question)
    if path == "refuse":
        return {"path": path, "answer": "That isn't in the data.", "sql": None, "sources": []}
    if path == "sql":
        run = run_sql(question)
        if usable(run):
            return {"path": path, "answer": sql_answer(question, run["df"]), "sql": run["sql"], "sources": []}
        answer, sources = answer_vector(question)
        return {"path": "sql>vector", "answer": answer, "sql": run["sql"], "sources": sources}
    answer, sources = answer_vector(question)
    return {"path": path, "answer": answer, "sql": None, "sources": sources}

In [ ]:
for q in [
    "Who won the 2021 Abu Dhabi Grand Prix?",
    "How did Max Verstappen perform at Monaco?",
    "Tell me about the 2021 Abu Dhabi Grand Prix",
    "What was the weather at the 2021 Monaco Grand Prix?",
]:
    result = ask(q)
    print(f"[{result['path']}] {q}\n  {result['answer']}\n")

## Evaluation

The `test3` questions were written after this router was committed, and nothing here changed after seeing them. Each pipeline is scored on its final text.

In [ ]:
REFUSAL = re.compile(r"(not|isn't) in the (context|database|data)|no usable result|no information", re.IGNORECASE)


def text_is_right(q: dict, answer: str) -> bool:
    values = q.get("accept", [])
    if q["truth_sql"]:
        values = [con.execute(q["truth_sql"]).fetchone()[0]] + values
    if not values:
        return bool(REFUSAL.search(answer))  # an unanswerable question should be refused
    for value in values:
        if isinstance(value, str) and value.lower() in answer.lower():
            return True
        if not isinstance(value, str) and any(abs(n - float(value)) <= 0.01 for n in numbers_in(answer)):
            return True
    return False


TEST3 = [q for q in QUESTIONS if q["split"] == "test3"]
rows = []
for q in TEST3:
    hybrid = ask(q["question"])
    sql_text = answer_sql_only(q["question"])
    vector_text = answer_vector(q["question"])[0]
    rows.append({"id": q["id"], "kind": q["kind"], "question": q["question"], "path": hybrid["path"],
                 "hybrid_right": text_is_right(q, hybrid["answer"]), "sql_right": text_is_right(q, sql_text),
                 "vector_right": text_is_right(q, vector_text), "hybrid": hybrid["answer"],
                 "sql": sql_text, "vector": vector_text})

for name in ("sql", "vector", "hybrid"):
    print(f"{name:<7} {sum(r[f'{name}_right'] for r in rows)}/{len(rows)}")
for r in rows:
    print(f"{r['id']} {r['kind']:<13} {r['path']:<11} sql={r['sql_right']!s:<5} vector={r['vector_right']!s:<5} hybrid={r['hybrid_right']!s:<5} {r['hybrid'][:90]!r}")

out = PROJECT_ROOT / "eval" / "results_nb06.json"
out.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")